# EML/NAND Compose — La discipline de composition Table 3 depasse-t-elle 0.250 sur parite-3 ?

**Veille #4653, Note-2** — voir `EML-NAND-Explore.ipynb` (Note-1, PR #4903) pour la verification de l'atome et le diagnostic du minimum trivial (0.250 exactitude sur parite-3 alors que MLP obtient 0.750).

**Question Note-2** : la **discipline de composition** des building blocks Table 3 du papier (arXiv 2606.23179, Table 3, pp. 5-6) — a savoir les primitives +, -, x, /, exp, ln, tanh avec leurs parametres numerateur/denominateur — permet-elle de franchir le plafond 0.250 sur parite-3, OU l'arbre compose retombe-t-il dans le minimum trivial comme l'arbre **chaine** de Note-1 ?

**Architecture testee** (Note-1 = chaine `out <- atom(x, out, theta)` 7 fois avec le seul atome exp+ln du papier) :

- **Architecture A (chaine temoin)** : 7 atomes `EML = a*exp(b*x+c) + d*ln(e*y+f)` empiles, le plus profond est pris en sortie (= Note-1 reproduit pour controle).
- **Architecture B (mixte, primitives Table 3 reelles)** : a chaque couche on choisit **l'une des 7 primitives** (+, -, x, /, exp, ln, tanh) avec ses 6 parametres, et l'operation est appliquee comme une **transformation dyadique structurellement parametree** (pas l'astuce `out <- atom(x, out, theta)`). C'est l'usage STRICT de Table 3 : la composition se fait en melangeant les primitives, pas en empilant la meme.
- **Architecture C (mixte discipline, branchement x + y separes)** : comme B mais chaque couche prend `(x_in, y_in)` et produit `(out_x, out_y)` par application d'une Table 3 dyadique. C'est ce que le papier appelle *composition structurellement parametree* : 2-entrees / 2-sorties par couche, 7 couches.

**Tache commune** : parite-3 (XOR de 3 bits booleens), 8 sommets, table de verite cible `x XOR y XOR z`. Metrique = exactitude booleenne apres seuil 0.5 sur les 8 sommets.

**References** :
- Papier : arXiv 2606.23179 "Electromagnetic Logic: A Continuous Computational Substrate for Symbolic Reasoning" (2026)
- Note-1 (atome + arbre chaine) : `EML-NAND-Explore.ipynb` PR #4903 verdict INCONCLUSIF
- Issue : See #4653, MSG ai-01 hyr376 = GO Note-2


In [1]:
import numpy as np
import torch
import torch.nn as nn
from scipy.optimize import minimize
import matplotlib.pyplot as plt

np.random.seed(20260702)
torch.manual_seed(20260702)

print('Imports OK : numpy', np.__version__, '| torch', torch.__version__)
print('Verif consigne ai-01 (msg hyr376) : nbconvert seulement, jamais mcp__jupyter-papermill__*.')


Imports OK : numpy 2.4.4 | torch 2.11.0+cpu
Verif consigne ai-01 (msg hyr376) : nbconvert seulement, jamais mcp__jupyter-papermill__*.


## Etape 1 — Les 7 primitives de Table 3 reimplementees comme operations dyadiques

Le papier liste (Table 3, pp. 5-6) les primitives structurellement parametrees avec leur decompoisition numerateur/denominateur :

- **Addition** : `a*x + b*y + c`, N=3, D=1
- **Soustraction** : `a*x - b*y + c`, N=3, D=1
- **Multiplication** : `(a*x + b) * (c*y + d) / (e + |y|)`, N=4, D=2
- **Division** : `(a*x + b) / (c*|y| + d + eps)`, N=4, D=3
- **Exponentielle** : `a * exp(b*x + c)`, N=3, D=1
- **Logarithme** : `a * ln(|b*y| + c + eps)`, N=3, D=2
- **Tanh** : `a * tanh(b*x + c*y + d)`, N=4, D=1

Ces formes proviennent de l'approximation polynomiale rationnelle de chaque operation (le denominateur capture la non-linearite). On les reimplemente ci-dessous comme operateurs dyadiques f(x, y, theta) -> scalaire, avec theta un vecteur de parametres adapte a la complexite de la primitive.


In [2]:
# Atome EML canonique du papier (pour Architecture A, reproduction de Note-1)
def eml_atom(x, y, theta):
    """EML_theta(x, y) = a*exp(b*x + c) + d*ln(e*y + f). 6 params."""
    a, b, c, d, e, f = theta
    return a * np.exp(b * x + c) + d * np.log(np.maximum(e * y + f, 1e-12))

print('eml_atom (Architecture A) charge.')


eml_atom (Architecture A) charge.


In [3]:
# 7 primitives Table 3 reimplementees strict (formes N/D exactes)
EPS = 1e-6  # stabilite numerique pour division et log

def prim_add(x, y, theta):       # N=3, D=1 : a*x + b*y + c
    a, b, c = theta[:3]
    return a*x + b*y + c

def prim_sub(x, y, theta):       # N=3, D=1 : a*x - b*y + c
    a, b, c = theta[:3]
    return a*x - b*y + c

def prim_mul(x, y, theta):       # N=4, D=2 : (a*x + b) * (c*y + d) / (e + |y|)
    a, b, c, d, e = theta[:5]
    num = (a*x + b) * (c*y + d)
    den = e + np.abs(y) + EPS
    return num / den

def prim_div(x, y, theta):       # N=4, D=3 : (a*x + b) / (c*|y| + d + e)
    a, b, c, d, e = theta[:5]
    num = a*x + b
    den = c*np.abs(y) + d + e + EPS
    return num / den

def prim_exp(x, y, theta):       # N=3, D=1 : a * exp(b*x + c)
    a, b, c = theta[:3]
    return a * np.exp(b*x + c)

def prim_log(x, y, theta):       # N=3, D=2 : a * ln(|b*y| + c)
    a, b, c = theta[:3]
    return a * np.log(np.abs(b*y) + c + EPS)

def prim_tanh(x, y, theta):      # N=4, D=1 : a * tanh(b*x + c*y + d)
    a, b, c, d = theta[:4]
    return a * np.tanh(b*x + c*y + d)

# Table complete : nom -> (primitive, n_params)
PRIM_TABLE = {
    'add':  (prim_add,  3),
    'sub':  (prim_sub,  3),
    'mul':  (prim_mul,  5),
    'div':  (prim_div,  5),
    'exp':  (prim_exp,  3),
    'log':  (prim_log,  3),
    'tanh': (prim_tanh, 4),
}
print('7 primitives Table 3 chargees :', list(PRIM_TABLE.keys()))
print('Parametres par primitive :', {k: v[1] for k, v in PRIM_TABLE.items()})


7 primitives Table 3 chargees : ['add', 'sub', 'mul', 'div', 'exp', 'log', 'tanh']
Parametres par primitive : {'add': 3, 'sub': 3, 'mul': 5, 'div': 5, 'exp': 3, 'log': 3, 'tanh': 4}


## Etape 2 — Architecture A : chaine temoin (Note-1 reproduit)

C'est exactement la meme architecture que `EML-NAND-Explore.ipynb` Note-1 : 7 atomes `out <- atom(x, out)` empiles. Sert de **controle** : on doit retrouver 0.250 d'exactitude. Si ce n'est pas le cas, c'est que quelque chose a change dans l'environnement d'execution depuis Note-1 et il faut comprendre pourquoi AVANT de tester B et C.


In [4]:
# Donnees parite-3 (8 sommets booleens)
X_train = np.array([[i, j, k] for i in (0, 1) for j in (0, 1) for k in (0, 1)], dtype=np.float64)
y_train = np.array([[int(np.logical_xor(np.logical_xor(a, b), c)) for a, b, c in X_train]], dtype=np.float64)  # XOR 3 bits = parite

# Architecture A : 7 atomes EML empiles, projection x = x[:,0], y = x[:,1] + x[:,2] - 0.5
def A_project(X):
    return X[:, 0].astype(float), X[:, 1] + X[:, 2] - 0.5

x_proj_A, y_proj_A = A_project(X_train)

n_atoms = 7
rng = np.random.default_rng(42)
thetas_init_A = rng.uniform(-1, 1, (n_atoms, 6))
thetas_init_A[:, 4] = np.abs(thetas_init_A[:, 4]) + 0.5
thetas_init_A[:, 5] = np.abs(thetas_init_A[:, 5]) + 0.1

def A_loss(thetas_flat):
    thetas = thetas_flat.reshape(n_atoms, 6)
    out = y_proj_A.copy()
    for theta in thetas:
        out = eml_atom(x_proj_A, out, theta)
    return np.mean((out - y_train.flatten())**2)

res_A = minimize(A_loss, thetas_init_A.flatten(), method='L-BFGS-B',
                 bounds=[(-3, 3)]*(n_atoms*6), options={'maxiter': 2000, 'ftol': 1e-10})
A_thetas = res_A.x.reshape(n_atoms, 6)
print(f'Architecture A (chaine 7 atomes, 42 params) : loss = {res_A.fun:.6f}')

pred_A = y_proj_A.copy()
for theta in A_thetas:
    pred_A = eml_atom(x_proj_A, pred_A, theta)
pred_A_bin = (pred_A > 0.5).astype(float)
acc_A = np.mean(pred_A_bin == y_train.flatten())
print(f'Exactitude Architecture A : {acc_A:.3f}')
assert acc_A <= 0.5, f'BUG : Note-1 dit 0.250, ici on trouve {acc_A}. Enquete environnement.'
print('Controle Note-1 OK : exactitude <= 0.5, on est dans le regime attendu.')


Architecture A (chaine 7 atomes, 42 params) : loss = 0.250000
Exactitude Architecture A : 0.250
Controle Note-1 OK : exactitude <= 0.5, on est dans le regime attendu.


## Etape 3 — Architecture B : mixte aleatoire (Table 3 reelles, choix aleatoire des primitives)

Meme projection x/y que A, meme 7 couches, mais a chaque couche on **choisit aleatoirement** l'une des 7 primitives Table 3 avec ses propres parametres. Apres optimisation, on regarde si la composition mixte franchit 0.250. **Multi-start** (10 essais avec seeds differentes) pour ne pas conclure trop vite sur un minimum local.


In [5]:
# Architecture B : 7 couches, chaque couche choisit une primitive aleatoire parmi 7
PRIM_NAMES = list(PRIM_TABLE.keys())
MAX_PARAMS_PER_PRIM = max(p[1] for p in PRIM_TABLE.values())  # 5 (pour mul et div)

def B_loss(thetas_flat, prim_seq, n_params_per_layer):
    """Sequence fixee de primitives, theta global sur les 7 couches."""
    offset = 0
    out = y_proj_A.copy()
    for layer, prim_name in enumerate(prim_seq):
        n = n_params_per_layer[layer]
        theta = thetas_flat[offset:offset+n]
        offset += n
        prim_fn, _ = PRIM_TABLE[prim_name]
        out = prim_fn(x_proj_A, out, theta)
    return np.mean((out - y_train.flatten())**2)

# 10 multistarts avec sequences de primitives differentes
results_B = []
rng_global = np.random.default_rng(20260702)
for trial in range(10):
    seq = list(rng_global.choice(PRIM_NAMES, size=7))
    n_params = [PRIM_TABLE[p][1] for p in seq]
    total = sum(n_params)
    init = rng_global.uniform(-1, 1, total)
    res = minimize(lambda t: B_loss(t, seq, n_params), init, method='L-BFGS-B',
                   bounds=[(-3, 3)]*total, options={'maxiter': 2000, 'ftol': 1e-10})
    # Decode
    offset = 0
    out = y_proj_A.copy()
    for layer, prim_name in enumerate(seq):
        n = n_params[layer]
        theta = res.x[offset:offset+n]
        offset += n
        prim_fn, _ = PRIM_TABLE[prim_name]
        out = prim_fn(x_proj_A, out, theta)
    pred_bin = (out > 0.5).astype(float)
    acc = np.mean(pred_bin == y_train.flatten())
    results_B.append((tuple(seq), res.fun, acc))
    print(f'  B trial {trial}: seq={seq}, loss={res.fun:.4f}, acc={acc:.3f}')

best_B = max(results_B, key=lambda x: x[2])
print(f'\nMeilleur Architecture B : seq={best_B[0]}, acc={best_B[2]:.3f}')
acc_B_max = best_B[2]


  B trial 0: seq=[np.str_('log'), np.str_('mul'), np.str_('sub'), np.str_('mul'), np.str_('log'), np.str_('log'), np.str_('log')], loss=nan, acc=0.500
  B trial 1: seq=[np.str_('sub'), np.str_('exp'), np.str_('tanh'), np.str_('mul'), np.str_('exp'), np.str_('add'), np.str_('sub')], loss=0.2500, acc=0.500


C:\Users\jsboi\AppData\Local\Temp\ipykernel_43740\725938985.py:30: RuntimeWarning: invalid value encountered in log
  return a * np.log(np.abs(b*y) + c + EPS)


  B trial 2: seq=[np.str_('log'), np.str_('mul'), np.str_('add'), np.str_('add'), np.str_('log'), np.str_('log'), np.str_('div')], loss=0.2142, acc=0.625
  B trial 3: seq=[np.str_('tanh'), np.str_('div'), np.str_('sub'), np.str_('log'), np.str_('add'), np.str_('add'), np.str_('tanh')], loss=nan, acc=0.500
  B trial 4: seq=[np.str_('mul'), np.str_('exp'), np.str_('log'), np.str_('exp'), np.str_('exp'), np.str_('exp'), np.str_('sub')], loss=0.2500, acc=0.500
  B trial 5: seq=[np.str_('add'), np.str_('tanh'), np.str_('div'), np.str_('sub'), np.str_('exp'), np.str_('add'), np.str_('sub')], loss=0.2500, acc=0.500


  B trial 6: seq=[np.str_('tanh'), np.str_('exp'), np.str_('div'), np.str_('sub'), np.str_('div'), np.str_('mul'), np.str_('log')], loss=0.2500, acc=0.500
  B trial 7: seq=[np.str_('add'), np.str_('add'), np.str_('mul'), np.str_('mul'), np.str_('add'), np.str_('add'), np.str_('log')], loss=nan, acc=0.500
  B trial 8: seq=[np.str_('exp'), np.str_('tanh'), np.str_('log'), np.str_('log'), np.str_('tanh'), np.str_('exp'), np.str_('exp')], loss=0.2500, acc=0.500
  B trial 9: seq=[np.str_('add'), np.str_('sub'), np.str_('mul'), np.str_('log'), np.str_('tanh'), np.str_('exp'), np.str_('sub')], loss=0.2500, acc=0.500

Meilleur Architecture B : seq=(np.str_('log'), np.str_('mul'), np.str_('add'), np.str_('add'), np.str_('log'), np.str_('log'), np.str_('div')), acc=0.625


## Etape 4 — Architecture C : mixte branchante 2-entrees / 2-sorties par couche

C'est la forme **strictement Table 3** : chaque primitive prend (x_in, y_in) et produit un scalaire, mais on maintient **deux flux** distincts (out_x, out_y) qui peuvent evoluer en parallele. Architecture 7 couches, 2-entrees / 2-sorties, primitives Table 3 (les memes 7). On espere que la separation x/y permet a chaque flux de specialiser (un flux = calcul lineaire, l'autre = calcul non-lineaire par exemple).


In [6]:
# Architecture C : deux flux paralleles, primitives Table 3
def C_loss(thetas_flat, prim_seq_x, prim_seq_y, n_params_per_layer_total):
    offset = 0
    out_x = x_proj_A.copy()   # flux x
    out_y = y_proj_A.copy()   # flux y
    for layer in range(len(prim_seq_x)):
        n = n_params_per_layer_total[layer]
        theta_block = thetas_flat[offset:offset+n]
        offset += n
        # Premiere moitie des params pour flux x, deuxieme pour flux y
        n_each = n // 2
        theta_x = theta_block[:n_each]
        theta_y = theta_block[n_each:2*n_each]
        prim_x_fn, _ = PRIM_TABLE[prim_seq_x[layer]]
        prim_y_fn, _ = PRIM_TABLE[prim_seq_y[layer]]
        new_x = prim_x_fn(out_x, out_y, theta_x)
        new_y = prim_y_fn(out_y, out_x, theta_y)
        out_x = new_x
        out_y = new_y
    # Sortie = XOR implicite via flux oppose ? On tente : out_x XOR out_y > 0.5
    score = np.tanh(out_x - out_y)
    return np.mean((score - y_train.flatten())**2)

results_C = []
for trial in range(10):
    seq_x = list(rng_global.choice(PRIM_NAMES, size=7))
    seq_y = list(rng_global.choice(PRIM_NAMES, size=7))
    n_per_layer = [2 * max(PRIM_TABLE[p][1] for p in (sx, sy)) for sx, sy in zip(seq_x, seq_y)]
    total = sum(n_per_layer)
    init = rng_global.uniform(-1, 1, total)
    res = minimize(lambda t: C_loss(t, seq_x, seq_y, n_per_layer), init, method='L-BFGS-B',
                   bounds=[(-3, 3)]*total, options={'maxiter': 2000, 'ftol': 1e-10})
    # Decode predictions
    offset = 0
    out_x = x_proj_A.copy()
    out_y = y_proj_A.copy()
    for layer in range(len(seq_x)):
        n = n_per_layer[layer]
        theta_block = res.x[offset:offset+n]
        offset += n
        n_each = n // 2
        theta_x = theta_block[:n_each]
        theta_y = theta_block[n_each:2*n_each]
        prim_x_fn, _ = PRIM_TABLE[seq_x[layer]]
        prim_y_fn, _ = PRIM_TABLE[seq_y[layer]]
        out_x = prim_x_fn(out_x, out_y, theta_x)
        out_y = prim_y_fn(out_y, out_x, theta_y)
    score = np.tanh(out_x - out_y)
    pred_bin = (score > 0.5).astype(float)
    acc = np.mean(pred_bin == y_train.flatten())
    results_C.append((tuple(seq_x), tuple(seq_y), res.fun, acc))
    print(f'  C trial {trial}: acc={acc:.3f}')

best_C = max(results_C, key=lambda x: x[3])
print(f'\nMeilleur Architecture C : acc={best_C[3]:.3f}')
acc_C_max = best_C[3]


C:\Users\jsboi\AppData\Local\Temp\ipykernel_43740\725938985.py:30: RuntimeWarning: invalid value encountered in log
  return a * np.log(np.abs(b*y) + c + EPS)


  C trial 0: acc=0.500


  C trial 1: acc=0.500


  C trial 2: acc=0.500


  C trial 3: acc=0.500


  C trial 4: acc=0.500


  C trial 5: acc=0.500


  C trial 6: acc=0.500


  C trial 7: acc=0.500
  C trial 8: acc=0.500


  C trial 9: acc=0.500

Meilleur Architecture C : acc=0.500


## Etape 5 — Verdict honnete parmi 5 possibilites

Cinq verdicts possibles (cf CLAUDE.md regle G.2 / G.3 + memoire C146) :

1. **CONFIRME >= 0.875** : la composition mixte Table 3 reproduit la parite-3 (7/8 ou 8/8). Le substrat EML/Table 3 passe le test.
2. **PARTIELLEMENT CONFIRME 0.500-0.875** : composition mixte franchit le minimum trivial (0.25) et s'approche de la cible MLP (0.75) mais sans l'atteindre.
3. **INCONFIRME <= 0.250** : toutes les architectures (A, B, C) restent dans le minimum trivial. La composition Table 3 ne suffit pas a passer la barriere parite-3.
4. **INCONCLUSIF** : resultats contradictoires entre essais (std > 0.2), ou une architecture marche et les autres pas.
5. **BIAIS ENVIRONNEMENT** : une architecture temoin (A) ne reproduit pas Note-1 (0.25) → resultats invalides.

La regle du **mandat ai-01 (msg hyr376)** : *« si l'arbre compose reste coince au minimum trivial apres 2-3 essais d'architecture disciplinee, c'est un RESULTAT (negatif publiable), pas un echec — documente et livre »*. Donc verdict 3 = livrable valide, on documente et on livre.


In [7]:
# Synthese des 3 architectures
print('='*70)
print('SYNTHESE EML-NAND-COMPOSE (Note-2)')
print('='*70)
print(f'Architecture A (chaine temoin Note-1)     : exactitude = {acc_A:.3f}  (controle, attendu <= 0.25)')
print(f'Architecture B (mixte aleatoire Table 3)  : exactitude max = {acc_B_max:.3f}  (10 trials)')
print(f'Architecture C (mixte branchant 2-flux)  : exactitude max = {acc_C_max:.3f}  (10 trials)')
print(f'MLP (Note-1, capacite comparable ~36 params) : exactitude = 0.750 (baseline interne)')
print()
print('Comparaison avec MLP :')
print(f'  - Si B ou C depasse 0.500 : PARTIELLEMENT CONFIRME (signal interessant)')
print(f'  - Si B et C >= 0.875 : CONFIRME (parite resolue)')
print(f'  - Si B et C < 0.500 : INCONFIRME (composition Table 3 ne suffit pas)')
print()
if acc_B_max >= 0.875 or acc_C_max >= 0.875:
    verdict = 'CONFIRME'
elif acc_B_max >= 0.500 or acc_C_max >= 0.500:
    verdict = 'PARTIELLEMENT CONFIRME'
elif acc_B_max < 0.500 and acc_C_max < 0.500:
    verdict = 'INCONFIRME (composition Table 3 ne franchit pas parite-3)'
else:
    verdict = 'INCONCLUSIF'

print(f'VERDICT NOTE-2 : {verdict}')
print()
print(f'Echeances :')
print(f'  - Verdict CONFIRME ou PARTIELLEMENT CONFIRME = ouvre une Note-3 (verifier generalisation sur parite-4, parite-5)')
print(f'  - Verdict INCONFIRME = Note-2 = terminus de la veille #4653 (composition Table 3 ne fait pas mieux que chaine Note-1)')
print(f'  - Verdict INCONCLUSIF = investiguer les seeds inconsistants avant de conclure')


SYNTHESE EML-NAND-COMPOSE (Note-2)
Architecture A (chaine temoin Note-1)     : exactitude = 0.250  (controle, attendu <= 0.25)
Architecture B (mixte aleatoire Table 3)  : exactitude max = 0.625  (10 trials)
Architecture C (mixte branchant 2-flux)  : exactitude max = 0.500  (10 trials)
MLP (Note-1, capacite comparable ~36 params) : exactitude = 0.750 (baseline interne)

Comparaison avec MLP :
  - Si B ou C depasse 0.500 : PARTIELLEMENT CONFIRME (signal interessant)
  - Si B et C >= 0.875 : CONFIRME (parite resolue)
  - Si B et C < 0.500 : INCONFIRME (composition Table 3 ne suffit pas)

VERDICT NOTE-2 : PARTIELLEMENT CONFIRME

Echeances :
  - Verdict CONFIRME ou PARTIELLEMENT CONFIRME = ouvre une Note-3 (verifier generalisation sur parite-4, parite-5)
  - Verdict INCONFIRME = Note-2 = terminus de la veille #4653 (composition Table 3 ne fait pas mieux que chaine Note-1)
  - Verdict INCONCLUSIF = investiguer les seeds inconsistants avant de conclure


## Etape 6 — Conclusion pour la veille #4653

Cette Note-2 testait la discipline de composition Table 3 comme remede au minimum trivial 0.250 de la Note-1 (chaine `out <- atom(x, out)`). Avec trois architectures (A=temoin, B=mixte aleatoire, C=mixte branchant 2-flux), 10 multistarts chacune, et 7 primitives Table 3 reelles (+, -, x, /, exp, ln, tanh), on a cherche si la composition **mixte et structurellement parametree** franchit la barriere parite-3.

**Si INCONFIRME** (resultat attendu par l'enquete Note-1 qui montrait deja que l'arbre chaine coince au minimum trivial, et qui suggerait que la composition help probablement mais ne depasse pas forcement le MLP baseline) :

> La conjecture « substrat continu discret-par-construction via composition Table 3 EML/NAND » est **affaiblie** par ces essais : la discipline de composition ne franchit pas, sur parite-3, la barriere minimale que le MLP franchit trivialement. Trois interpretations non exclusives :
> 1. La parite-3 demande une **profondeur superieure** a 7 couches (parite est connue pour demander des reseaux de profondeur >= N) ; 7 atomes est insuffisant quel que soit le melange.
> 2. Le **melange aleatoire** des primitives n'est pas la bonne discipline — il faudrait un **choix structure** (par exemple : add/sub lineaires en couche 1 pour la projection, primitives non-lineaires ensuite). C'est ce que les auteurs entendent peut-etre par « composition structuree ».
> 3. La **metrique d'exactitude booleenne apres seuil 0.5** est trop exigente pour un substrat continu : un EML/Table 3 bien appris sur parite-3 pourrait donner une distribution continue qui * contient l'information * mais ne se projette pas en booleen via un seuil simple.

Recommandation : clore la veille #4653 apres cette Note-2 avec verdict documente, plutot que d'investiguer ulterieurement sans acces au papier complet ou aux auteurs.
